# Ethiopian Address Parser v2
## ML-based approach: XLM-RoBERTa NER + FAISS Vector Search

**Problem:** Convert free-text Ethiopian addresses (Amharic/English/mixed) to GPS coordinates.

**Pipeline:**
1. Normalize text (Amharic homophone normalization, script detection)
2. NER: Extract entities (subcity, landmark, direction) using XLM-RoBERTa
3. Match landmarks against FAISS vector index using multilingual embeddings
4. Return structured result with GPS coordinates

**Models used:**
- `mbeukman/xlm-roberta-base-finetuned-ner-amharic` — Amharic NER (93% F1)
- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` — Multilingual embeddings

**References:**
- ANEC: Amharic NER corpus (arxiv 2207.00785)
- MasakhaNER: African language NER (ACL 2021)
- GeoAgent: LLM + geospatial tools (ACL 2024)

In [ ]:
# Install dependencies (Kaggle/Colab)
!pip install -q transformers sentencepiece torch faiss-cpu rapidfuzz sentence-transformers

In [ ]:
# Clone the repo
!git clone -b v2-ml-parser https://github.com/RealMati/EAP.git 2>/dev/null || (cd EAP && git pull)
import sys
sys.path.insert(0, 'EAP')
%cd EAP

## 1. Test Rule-Based Only (Baseline)

In [ ]:
from eap.parser import EthiopianAddressParser

# Baseline: rule-based NER, fuzzy string matching
parser_baseline = EthiopianAddressParser(
    data_dir='.',
    use_transformer_ner=False,
    use_semantic_search=False,
)
parser_baseline.load()

In [ ]:
# Quick test
test_addresses = [
    'behind Edna Mall, Bole',
    'ቦሌ ኤድና ሞል ጀርባ',
    'kirkos hilton hotel aTeGeb',
    'ኪርቆስ ሂልተን ሆቴል አጠገብ',
    'meskel square area',
    'መስቀል አደባባይ ቅርብ',
    'bole medhanialem church',
    'ቦሌ መድሃኔዓለም ቤተክርስቲያን',
    'near friendship hotel, bole',
    'ቦሌ ፍሬንድሽፕ ሆቴል ፊት ለፊት',
]

print('BASELINE (Rule-based only)')
print('=' * 80)
for addr in test_addresses:
    r = parser_baseline.parse(addr)
    print(f'Input:    {addr}')
    print(f'Subcity:  {r.subcity or "(none)"}')
    print(f'Landmark: {r.landmark_name or "(none)"} ({r.landmark_match_method}, {r.landmark_match_score:.0f}%)')
    print(f'Coords:   {r.latitude}, {r.longitude}')
    print(f'Conf:     {r.confidence:.0f}%')
    print()

## 2. Test with Transformer NER

In [ ]:
# ML NER: XLM-RoBERTa fine-tuned on Amharic
parser_ml_ner = EthiopianAddressParser(
    data_dir='.',
    use_transformer_ner=True,
    use_semantic_search=False,
)
parser_ml_ner.load()

In [ ]:
print('WITH TRANSFORMER NER')
print('=' * 80)
for addr in test_addresses:
    r = parser_ml_ner.parse(addr)
    ner_landmarks = r.ner_result.landmarks if r.ner_result else []
    print(f'Input:     {addr}')
    print(f'NER found: {ner_landmarks}')
    print(f'Subcity:   {r.subcity or "(none)"}')
    print(f'Landmark:  {r.landmark_name or "(none)"} ({r.landmark_match_method}, {r.landmark_match_score:.0f}%)')
    print(f'Coords:    {r.latitude}, {r.longitude}')
    print(f'Conf:      {r.confidence:.0f}%')
    print()

## 3. Test with Semantic Search (FAISS + Embeddings)

In [ ]:
# Full pipeline: Transformer NER + Semantic search
parser_full = EthiopianAddressParser(
    data_dir='.',
    use_transformer_ner=True,
    use_semantic_search=True,
)
parser_full.load()

In [ ]:
print('FULL PIPELINE (Transformer NER + Semantic Search)')
print('=' * 80)
for addr in test_addresses:
    r = parser_full.parse(addr)
    print(f'Input:     {addr}')
    print(f'Subcity:   {r.subcity or "(none)"}')
    print(f'Landmark:  {r.landmark_name or "(none)"} ({r.landmark_match_method}, {r.landmark_match_score:.0f}%)')
    print(f'Coords:    {r.latitude}, {r.longitude}')
    print(f'Conf:      {r.confidence:.0f}%')
    print(f'Candidates: {[(c.landmark.name, f"{c.score:.0f}%", c.method) for c in r.candidates[:3]]}')
    print()

## 4. Full Test Suite — Compare All Modes

In [ ]:
import pandas as pd

def load_tests(filepath):
    tests = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = [p.strip() for p in line.split('|')]
            if len(parts) >= 3:
                tests.append((parts[0], parts[1], parts[2]))
    return tests

def evaluate(parser, tests, mode_name):
    rows = []
    for addr, exp_sc, exp_lm in tests:
        r = parser.parse(addr)
        sc_ok = exp_sc.lower() in (r.subcity or '').lower() if exp_sc != 'unknown' else True
        lm_ok = exp_lm.lower() in (r.landmark_name or '').lower() if exp_lm != 'unknown' else True
        rows.append({
            'mode': mode_name,
            'address': addr,
            'subcity_ok': sc_ok,
            'landmark_ok': lm_ok,
            'confidence': r.confidence,
            'method': r.landmark_match_method,
        })
    return pd.DataFrame(rows)

# Load all test files
categories = {
    'english': 'tests/addresses_english_only.txt',
    'transliterated': 'tests/addresses_transliterated.txt',
    'mixed': 'tests/addresses_mixed.txt',
    'amharic': 'tests/addresses_amharic_only.txt',
}

all_results = []
parsers = {
    'baseline': parser_baseline,
    'ml_ner': parser_ml_ner,
    'full': parser_full,
}

for cat, filepath in categories.items():
    tests = load_tests(filepath)
    for mode, p in parsers.items():
        df = evaluate(p, tests, mode)
        df['category'] = cat
        all_results.append(df)

results = pd.concat(all_results, ignore_index=True)

In [ ]:
# Summary table
summary = results.groupby(['mode', 'category']).agg(
    subcity_pct=('subcity_ok', lambda x: round(100 * x.mean(), 1)),
    landmark_pct=('landmark_ok', lambda x: round(100 * x.mean(), 1)),
    avg_confidence=('confidence', lambda x: round(x.mean(), 1)),
    count=('address', 'count'),
).reset_index()

print('\nRESULTS BY MODE AND CATEGORY')
print('=' * 80)
for mode in ['baseline', 'ml_ner', 'full']:
    mode_data = summary[summary['mode'] == mode]
    print(f'\n--- {mode.upper()} ---')
    for _, row in mode_data.iterrows():
        print(f"  {row['category']:<20} Subcity: {row['subcity_pct']:>5.1f}%  Landmark: {row['landmark_pct']:>5.1f}%  Conf: {row['avg_confidence']:>5.1f}%")
    overall_sc = round(100 * results[results['mode']==mode]['subcity_ok'].mean(), 1)
    overall_lm = round(100 * results[results['mode']==mode]['landmark_ok'].mean(), 1)
    overall_conf = round(results[results['mode']==mode]['confidence'].mean(), 1)
    print(f"  {'OVERALL':<20} Subcity: {overall_sc:>5.1f}%  Landmark: {overall_lm:>5.1f}%  Conf: {overall_conf:>5.1f}%")

## 5. Data Quality Analysis

In [ ]:
import json

# Analyze landmark data coverage
for fname in ['addis_landmarks.json', 'kilo_landmarks.json']:
    with open(fname) as f:
        data = json.load(f)
    total = len(data)
    has_am = sum(1 for d in data if d.get('amharic'))
    has_sc = sum(1 for d in data if d.get('subcity'))
    has_alias = sum(1 for d in data if d.get('aliases'))
    print(f'{fname}:')
    print(f'  Total: {total}')
    print(f'  With Amharic name: {has_am} ({100*has_am/total:.0f}%)')
    print(f'  With subcity: {has_sc} ({100*has_sc/total:.0f}%)')
    print(f'  With aliases: {has_alias} ({100*has_alias/total:.0f}%)')
    print()

## 6. Next Steps

If accuracy is still low, the highest-impact improvements are:

1. **Fill missing Amharic names** — 76% of landmarks have no Amharic name. Use OSM Overpass to extract `name:am` tags.
2. **Try Amharic-specific embeddings** — `RoBERTa-Base-Amharic-Embed` (ACL 2025) outperforms multilingual models by 17.6%.
3. **Fine-tune NER on dispatch data** — Annotate ~500 real dispatch addresses and fine-tune XLM-R.
4. **More landmark data** — Use OSM Overpass API to dump ALL POIs in Addis Ababa bounding box.